# Live lcls-ii-belt 

In [1]:
!echo $LCLS_LATTICE

/sdf/group/ad/sw/scm/repos/optics/lcls-lattice/


In [2]:
!caget ACCL:L1B:0200:AL:EACT

ACCL:L1B:0200:AL:EACT          158.025


In [3]:
!caget BPMS:GUNB:314:TMIT

BPMS:GUNB:314:TMIT             4.29975e+08


In [4]:
%load_ext autoreload
%autoreload 2

from tools import isotime

from belt.tools import NpEncoder
import pandas as pd
import numpy as np

import h5py
import json
import epics

import sys
import os
import toml
from time import sleep, time
import datetime
from belt.evaluate import default_belt_merit
from belt.belt_impact import run_belt, evaluate_belt
from make_dashboard import make_dashboard

import matplotlib.pyplot as plt

import matplotlib as mpl
mpl.use('Agg')

# Nicer plotting
%config InlineBackend.figure_format = 'retina'

In [5]:
# Saving and loading
def save_pvdata(filename, pvdata, isotime):
    with h5py.File(filename, 'w') as h5:
        h5.attrs['isotime'] = np.bytes_(isotime)
        for k, v in pvdata.items():
            if isinstance(v, str):
                v =  np.bytes_(v)
            h5[k] = v 
def load_pvdata(filename):
    
    if not os.path.exists(filename):
        raise ValueError(f'H5 file does not exist: {filename} ')
    pvdata = {}
    with h5py.File(filename, 'r') as h5:
        isotime = h5.attrs['isotime']
        for k in h5:
            v = np.array(h5[k])        
            if v.dtype.char == 'S':
                v = str(v.astype(str))
            pvdata[k] = v
            
    return pvdata, isotime

## Set parameters

In [6]:
phase_shift = 1.5        # shift in L1 and L2 phase
initial_energy = 75e6    # Energy of the input beam 
input_beam =  "/sdf/data/ad/ard/u/jytang/lume-belt-live/STCAV_data/particle-2025-04-05.h5"  # Input beam generated from STCAV image


## Read Live PV


In [7]:
CSV = 'pv_mapping/lclsii_belt.csv'
DF = pd.read_csv(CSV)#.dropna()

PVLIST = list(DF['device_pv_name'].dropna()) 


DF

,Variable,device_pv_name,pv_unit
0,Charge,BPMS:GUNB:314:TMIT,n_elecrons
1,Initial_energy,ACCL:L0B:0100:AACTMEANSUM,MV
2,L1B_energy,ACCL:L1B:0200:AL:EACT,MV
3,L1B_amp1,ACCL:L1B:0200:AL:CO_AMPL,MV
4,L1B_amp2,ACCL:L1B:0200:AL:NEG_AMPL,MV
5,L1B_amp3,ACCL:L1B:0200:AL:POS_AMPL,MV
6,L1B_chirp,ACCL:L1B:0200:AL:CACT,MV
7,L1B_phase,ACCL:L1B:0200:AL:BETA,deg
8,HL_amplitude,ACCL:L1B:0200:AL:NOTA_AMPL,MV
9,HL_phase,ACCL:L1B:0200:AL:NOTA_PHASE,deg


In [8]:
DF.loc[DF["Variable"] == "L1B_energy"] 

,Variable,device_pv_name,pv_unit
2,L1B_energy,ACCL:L1B:0200:AL:EACT,MV


In [9]:
LIVE = True
if LIVE:
    MONITOR = {pvname:epics.PV(pvname) for pvname in PVLIST}
   

In [10]:
MONITOR

{'BPMS:GUNB:314:TMIT': <PV 'BPMS:GUNB:314:TMIT', count=1, type=time_double, access=read-only>,
 'ACCL:L0B:0100:AACTMEANSUM': <PV 'ACCL:L0B:0100:AACTMEANSUM', count=1, type=time_double, access=read-only>,
 'ACCL:L1B:0200:AL:EACT': <PV 'ACCL:L1B:0200:AL:EACT', count=1, type=time_double, access=read-only>,
 'ACCL:L1B:0200:AL:CO_AMPL': <PV 'ACCL:L1B:0200:AL:CO_AMPL', count=1, type=time_double, access=read-only>,
 'ACCL:L1B:0200:AL:NEG_AMPL': <PV 'ACCL:L1B:0200:AL:NEG_AMPL', count=1, type=time_double, access=read-only>,
 'ACCL:L1B:0200:AL:POS_AMPL': <PV 'ACCL:L1B:0200:AL:POS_AMPL', count=1, type=time_double, access=read-only>,
 'ACCL:L1B:0200:AL:CACT': <PV 'ACCL:L1B:0200:AL:CACT', count=1, type=time_double, access=read-only>,
 'ACCL:L1B:0200:AL:BETA': <PV 'ACCL:L1B:0200:AL:BETA', count=1, type=time_double, access=read-only>,
 'ACCL:L1B:0200:AL:NOTA_AMPL': <PV 'ACCL:L1B:0200:AL:NOTA_AMPL', count=1, type=time_double, access=read-only>,
 'ACCL:L1B:0200:AL:NOTA_PHASE': <PV 'ACCL:L1B:0200:AL:NOT

In [11]:
def get_snapshot(snapshot_file=None):
        
    if LIVE:
        itime = isotime()
        pvdata =  {k:MONITOR[k].get() for k in MONITOR}
        
    else:
        pvdata, itime = load_pvdata(snapshot_file)
        itime = itime.decode('utf-8')
    
    #logger.info(f'Acquired settings from EPICS at: {itime}')
    
    epics_working_check = [val for val in pvdata.values() if val is None]
    
    if len(epics_working_check) == len(list(pvdata.keys())):
        raise Exception(f'EPICS returned None for all keys. Please check if you are able to connect to Accelerator')

    VCC_Key = None
    
    for k, v in pvdata.items():
        
        if v is None:
            raise ValueError(f'EPICS get for {k} returned None')
        
        if ':IMAGE:ARRAYDATA' in k.upper():
            VCC_Key = k
            found = False
            logger.info(f'Waiting for good {k}')
            counter = 0
            USE_VCC_LOCAL = True
            while not found and counter < 5:
                counter += 1
                if v is None:
                    continue
                if v.std() > 10:
                    found = True
                else:
                    v = MONITOR[k].get()
            if counter == 5:
                logger.info(f'VCC is not working. Defaulting to None.')
                USE_VCC_LOCAL = False
            elif np.ptp(v) < 128:
                v = v.astype(np.int8) # Downcast preemptively 
            pvdata[k] = v
        else:
            USE_VCC_LOCAL = False

    if not USE_VCC_LOCAL and VCC_Key in pvdata:
        del pvdata[VCC_Key]

    return pvdata, itime, USE_VCC_LOCAL

In [12]:
df = DF[DF['device_pv_name'].notna()]
assert len(df) > 0, 'Empty dataframe!'
    
pv_names = list(df['device_pv_name'])

pvdata, itime, USE_VCC_LOCAL = get_snapshot(None)
    
df['pv_value'] = [pvdata[k] for k in pv_names]

In [13]:
df

,Variable,device_pv_name,pv_unit,pv_value
0,Charge,BPMS:GUNB:314:TMIT,n_elecrons,4.268471e+08
1,Initial_energy,ACCL:L0B:0100:AACTMEANSUM,MV,9.493283e+01
2,L1B_energy,ACCL:L1B:0200:AL:EACT,MV,1.581753e+02
3,L1B_amp1,ACCL:L1B:0200:AL:CO_AMPL,MV,1.021434e+02
4,L1B_amp2,ACCL:L1B:0200:AL:NEG_AMPL,MV,7.290166e+01
5,L1B_amp3,ACCL:L1B:0200:AL:POS_AMPL,MV,8.231522e+01
6,L1B_chirp,ACCL:L1B:0200:AL:CACT,MV,-9.199902e+01
7,L1B_phase,ACCL:L1B:0200:AL:BETA,deg,-2.148235e+01
8,HL_amplitude,ACCL:L1B:0200:AL:NOTA_AMPL,MV,5.372890e+01
9,HL_phase,ACCL:L1B:0200:AL:NOTA_PHASE,deg,9.512199e+00


In [14]:
def get_settings(csv, base_settings={}, snapshot_dir=None, snapshot_file=None):
    """
    Fetches live settings for all devices in the CSV table, and translates them to simulation inputs
     
    """
    df = DF[DF['device_pv_name'].notna()]
    assert len(df) > 0, 'Empty dataframe!'
    
    pv_names = list(df['device_pv_name'])

    pvdata, itime, USE_VCC_LOCAL = get_snapshot(snapshot_file)
    
    df['pv_value'] = [pvdata[k] for k in pv_names]
    
    # Assign impact
    #df['impact_value'] = df['impact_factor']*df['pv_value'] 
    #if 'impact_offset' in df:
    #    df['impact_value'] = df['impact_value']  + df['impact_offset']

    # Collect settings
    settings = base_settings.copy()


    HL_phase = df.loc[df["Variable"] == "HL_phase", 'pv_value' ].values[0] - 180
    HL_amplitude = df.loc[df["Variable"] == "HL_amplitude", 'pv_value' ].values[0]*1e6
    HL_gradient = HL_amplitude/5.5346304
    
    #HL_energy = df.loc[df["Variable"] == "HL_energy", 'pv_value' ].values[0]
    #HL_chirp = df.loc[df["Variable"] == "HL_chirp", 'pv_value' ].values[0]

    #HL_phase = df.loc[df["Variable"] == "HL_phase", 'pv_value' ].values[0]
    #HL_amplitude = np.abs(HL_energy/np.cos(HL_phase/180*np.pi)*1e6/5.5346304)

    #HL_phase = df.loc[df["Variable"] == "HL_phase_2", 'pv_value' ].values[0]
    #HL_amplitude = df.loc[df["Variable"] == "HL_amplitude", 'pv_value' ].values[0]*1e6
    #HL_amplitude = (df.loc[df["Variable"] == "HL_amplitude_chirponly", 'pv_value' ].values[0] +
    #                df.loc[df["Variable"] == "HL_amplitude_+FBK", 'pv_value' ].values[0] +
    #                df.loc[df["Variable"] == "HL_amplitude_-FBK", 'pv_value' ].values[0] )*1e6
    #HL_gradient = HL_amplitude/5.5346304
    
    L1_energy = df.loc[df["Variable"] == "L1B_energy", 'pv_value' ].values[0]
    L1_chirp = df.loc[df["Variable"] == "L1B_chirp", 'pv_value' ].values[0]

    L1_phase = df.loc[df["Variable"] == "L1B_phase", 'pv_value' ].values[0]
    L1_amplitude = (df.loc[df["Variable"] == "L1B_amp1", 'pv_value' ].values[0] + 
                    df.loc[df["Variable"] == "L1B_amp2", 'pv_value' ].values[0] +
                    df.loc[df["Variable"] == "L1B_amp3", 'pv_value' ].values[0])*1e6
    L1_gradient = L1_amplitude/16.603888
                    
    #L1_amplitude = np.abs((L1_energy - HL_energy)/ np.cos(L1_phase/180*np.pi)*1e6/16.603888)
    #L1_amplitude = np.abs((L1_energy - HL_energy)*1e6/16.603888)



    L2_energy = df.loc[df["Variable"] == "L2B_energy", 'pv_value' ].values[0]
    L2_chirp = df.loc[df["Variable"] == "L2B_chirp", 'pv_value' ].values[0]

    L2_phase = df.loc[df["Variable"] == "L2B_phase", 'pv_value' ].values[0]  
    L2_gradient = np.abs(L2_energy/np.cos(L2_phase/180*np.pi)*1e6/99.623328)
   
    
    

    L3_energy = df.loc[df["Variable"] == "L3B_energy", 'pv_value' ].values[0]
    L3_chirp = df.loc[df["Variable"] == "L3B_chirp", 'pv_value' ].values[0]

    L3_phase = df.loc[df["Variable"] == "L3B_phase", 'pv_value' ].values[0]
    L3_gradient = np.abs(L3_energy/np.cos(L3_phase/180*np.pi)*1e6/166.038878)
    #L3_amplitude = np.abs(L3_energy*1e6/166.038878)
    
    
    BC1_energy = df.loc[df["Variable"] == "BC1_energy", 'pv_value' ].values[0]/1e3
    #BC1_rigidity = (df.loc[df["Variable"] == "BCX11", 'pv_value' ].values[0] + df.loc[df["Variable"] == "BCX12", 'pv_value' ].values[0] +
    #         df.loc[df["Variable"] == "BCX13", 'pv_value' ].values[0] + df.loc[df["Variable"] == "BCX14", 'pv_value' ].values[0])/4/10
    #BC1_angle = BC1_rigidity/(3.3356*BC1_energy)

    BC2_energy = df.loc[df["Variable"] == "BC2_energy", 'pv_value' ].values[0]/1e3
    #BC2_rigidity = (df.loc[df["Variable"] == "BCX21", 'pv_value' ].values[0] + df.loc[df["Variable"] == "BCX22", 'pv_value' ].values[0] +
    #         df.loc[df["Variable"] == "BCX23", 'pv_value' ].values[0] + df.loc[df["Variable"] == "BCX24", 'pv_value' ].values[0])/4/10
    #BC2_angle = BC2_rigidity/(3.3356*BC2_energy)


    BC1_rigidity = df.loc[df["Variable"] == "BCX11", 'pv_value' ].values[0] /10
    BC1_angle = BC1_rigidity/(3.3356*BC1_energy)

    BC2_rigidity = df.loc[df["Variable"] == "BCX21", 'pv_value' ].values[0]/10

    BC2_angle = BC2_rigidity/(3.3356*BC2_energy)


    #initial_energy = df.loc[df["Variable"] == "Initial_energy", 'pv_value'].values[0]*1e6

    BC1_energy_increment =BC1_energy*1e9 - (initial_energy + L1_amplitude*np.cos(L1_phase/180*np.pi) +  HL_amplitude*np.cos(HL_phase/180*np.pi))
    #BC1_energy_increment =BC1_energy*1e9 - (90e6 + L1_energy*1e6 )
    BC2_energy_increment = BC2_energy*1e9 - (BC1_energy*1e9 + L2_energy*1e6)

    settings["BC1:angle"] = BC1_angle
    settings["BC2:angle"] = BC2_angle
    settings["L1:gradient"] = L1_gradient
    settings["L1:phase_deg"] = L1_phase + phase_shift
    settings["L2:gradient"] = L2_gradient
    settings["L2:phase_deg"] = L2_phase + phase_shift
    settings["L3:gradient"] = L3_gradient
    settings["L3:phase_deg"] = L3_phase 
    settings["HL:gradient"] = HL_gradient
    settings["HL:phase_deg"] = HL_phase 
    settings["EBC1:energy_increment"] = BC1_energy_increment
    settings["EBC2:energy_increment"] = BC2_energy_increment
    
    #if DEBUG:
    #    settings['total_charge'] = 0
    #else:
    #    settings['total_charge'] = 1 # Will be updated with particles

    # VCC image
    #if USE_VCC_LOCAL:
    #    logger.info('Getting VCC Live Distgen')
    #    dfile, img, cutimg = get_live_distgen_xy_dist(filename=DISTGEN_LASER_FILE, vcc_device=VCC_DEVICE, pvdata=pvdata)  
    #    settings['distgen:xy_dist:file'] = dfile
    #elif USE_SAVED_VCC:
    #    settings['distgen:xy_dist:file'] = SAVED_VCC
    #    img, cutimg = None, None
    #else:
    #    img, cutimg = None, None
        #settings['distgen:r_dist:max_r:value'] = 0.35 # TEMP     
        
    if snapshot_dir and not snapshot_file:
        filename = os.path.abspath(os.path.join(snapshot_dir, f'{MODEL}-snapshot-{itime}.h5'))
    #    total_charge_pC = settings['distgen:total_charge:value']
    #    if total_charge_pC < MIN_CHARGE_pC:
    #        logger.info(f'total charge is too low: {total_charge_pC:.2f} pC, not saving snapshot')         
    #    else:
        save_pvdata(filename, pvdata, itime)
    #        logger.info(f'EPICS shapshot written: {filename}')
        
        
    return settings, df, itime

In [15]:
# Patch this into the function below for the dashboard creation
def my_merit(belt_object, itime):
    # Collect standard output statistics
    merit0 = default_belt_merit(belt_object)
    
    PLOT_OUTPUT_DIR_DATED = convertToDatedFormat(PLOT_OUTPUT_DIR)
    #Overriding at runtime to save in dated folders
    DASHBOARD_KWARGS["outpath"] = PLOT_OUTPUT_DIR_DATED
    
    # Make the dashboard from the evaluated object
    plot_file = make_dashboard(belt_object, itime=itime, **DASHBOARD_KWARGS)
    #print('Dashboard written:', plot_file)
    #logger.info(f'Dashboard written: {plot_file}')
    
    # Make all readable
    os.chmod(plot_file, 0o644)
    
    # Assign extra info
    merit0['plot_file'] = plot_file    
    merit0['isotime'] = itime
    
    # Clear any buffers
    plt.close('all')

    return merit0

In [16]:
def convertToDatedFormat(destionation_folder):
    curr_date = datetime.date.today()
    year,month,day = curr_date.strftime('%Y'),curr_date.strftime('%m'),curr_date.strftime('%d')
    destionation_folder_dated = destionation_folder + "/" + year + "/" + month + "/" + day

    if not os.path.exists(destionation_folder_dated):
        os.makedirs(destionation_folder_dated)
    
    return destionation_folder_dated

In [17]:
dat = {}

MODEL = 'LCLSII'
HOST = 's3df'
ARCHIVE_DIR = './archive'

SNAPSHOT_DIR = './snapshot'
SUMMARY_OUTPUT_DIR = './summary'
PLOT_OUTPUT_DIR = './plot'
#SETTINGS0 = {}
#SETTINGS0 = {"Impact_particles": "/sdf/data/ad/ard/u/jytang/lume-belt-live/impact_particles/final_particles.h5", "num_doublings": 7}
#SETTINGS0 = {"Impact_particles": "/sdf/data/ad/ard/u/jytang/lume-belt-live/STCAV_data/particle-2025-04-05.h5"}
SETTINGS0 = {"Impact_particles": input_beam}
CONFIG0 = {"input": "example/belt.in", "workdir": os.environ.get("SCRATCH")}
PREFIX = f'lume-belt-live-demo-{HOST}-{MODEL}'


DASHBOARD_KWARGS = {'outpath':PLOT_OUTPUT_DIR,            
                    'name' : PREFIX
                   }    

SNAPSHOT = None
SNAPSHOT_DIR_DATED = convertToDatedFormat(SNAPSHOT_DIR)
ARCHIVE_DIR_DATED = convertToDatedFormat(ARCHIVE_DIR)
SUMMARY_OUTPUT_DIR_DATED = convertToDatedFormat(SUMMARY_OUTPUT_DIR)
settings, df, itime = get_settings(CSV,
                                                           SETTINGS0,
                                                           snapshot_dir=SNAPSHOT_DIR_DATED,
                                                          snapshot_file=SNAPSHOT)       

In [18]:
0.0468343189785044**2*(9.86910678674 +2/3*0.549165215621)*2

0.04490094345151553

In [19]:
0.10089897660889431**2*(2.44781923989 +2/3*0.203558243263)*2

0.05260368182746314

In [20]:
settings

{'Impact_particles': '/sdf/data/ad/ard/u/jytang/lume-belt-live/STCAV_data/particle-2025-04-05.h5',
 'BC1:angle': np.float64(0.10131724822512624),
 'BC2:angle': np.float64(0.04686364343970674),
 'L1:gradient': np.float64(15499994.513108738),
 'L1:phase_deg': np.float64(-20.005675416223756),
 'L2:gradient': np.float64(14419313.801480396),
 'L2:phase_deg': np.float64(-22.07116405369782),
 'L3:gradient': np.float64(13582194.75073566),
 'L3:phase_deg': np.float64(0.0),
 'HL:gradient': np.float64(9707765.965052761),
 'HL:phase_deg': np.float64(-170.48780492068897),
 'EBC1:energy_increment': np.float64(-26546835.993656337),
 'EBC2:energy_increment': np.float64(-1608927.6896996498)}

In [21]:
settings

{'Impact_particles': '/sdf/data/ad/ard/u/jytang/lume-belt-live/STCAV_data/particle-2025-04-05.h5',
 'BC1:angle': np.float64(0.10131724822512624),
 'BC2:angle': np.float64(0.04686364343970674),
 'L1:gradient': np.float64(15499994.513108738),
 'L1:phase_deg': np.float64(-20.005675416223756),
 'L2:gradient': np.float64(14419313.801480396),
 'L2:phase_deg': np.float64(-22.07116405369782),
 'L3:gradient': np.float64(13582194.75073566),
 'L3:phase_deg': np.float64(0.0),
 'HL:gradient': np.float64(9707765.965052761),
 'HL:phase_deg': np.float64(-170.48780492068897),
 'EBC1:energy_increment': np.float64(-26546835.993656337),
 'EBC2:energy_increment': np.float64(-1608927.6896996498)}

In [22]:
def run1():
    dat = {}

    SNAPSHOT_DIR_DATED = convertToDatedFormat(SNAPSHOT_DIR)
    ARCHIVE_DIR_DATED = convertToDatedFormat(ARCHIVE_DIR)
    SUMMARY_OUTPUT_DIR_DATED = convertToDatedFormat(SUMMARY_OUTPUT_DIR)
        
    # Acquire settings
    mysettings, df,  itime = get_settings(CSV,
                                                           SETTINGS0,
                                                           snapshot_dir=SNAPSHOT_DIR_DATED,
                                                          snapshot_file=SNAPSHOT)        
    print(mysettings)
    dat['isotime'] = itime
    
    # Record inputs
    dat['inputs'] = mysettings
    dat['config'] = CONFIG0
    dat['pv_mapping_dataframe'] = df.to_dict()
    
    #logger.info(f'Running evaluate_impact_with_distgen...')

    t0 = time()
    
    #total_charge_pC = mysettings['distgen:total_charge:value']
    #if total_charge_pC < MIN_CHARGE_pC:
    #    logger.info(f'total charge is too low: {total_charge_pC:.2f} pC, skipping')
    #    return dat
    
    outputs = evaluate_belt(CONFIG0, mysettings,
                                       merit_f=lambda x: my_merit(x, itime),
                                       archive_path=ARCHIVE_DIR_DATED,
                                        verbose=True )
    
    dat['outputs'] =  outputs   
    #logger.info(f'...finished in {(time()-t0):.1f} s')
    fname = fname=f'{SUMMARY_OUTPUT_DIR_DATED}/{PREFIX}-{itime}.json'

    json.dump(dat, open(fname, 'w'), cls=NpEncoder)
    #logger.info(f'Summary output written: {fname}')
    return dat
    

In [23]:
#charge 75pC for 5/17/2025 shift
result = run1()

{'Impact_particles': '/sdf/data/ad/ard/u/jytang/lume-belt-live/STCAV_data/particle-2025-04-05.h5', 'BC1:angle': np.float64(0.10124061478887932), 'BC2:angle': np.float64(0.04686192354931788), 'L1:gradient': np.float64(15499999.757463621), 'L1:phase_deg': np.float64(-19.99214623827802), 'L2:gradient': np.float64(14409695.952180438), 'L2:phase_deg': np.float64(-21.94854548707396), 'L3:gradient': np.float64(13580757.857706686), 'L3:phase_deg': np.float64(0.0), 'HL:gradient': np.float64(9707766.095875073), 'HL:phase_deg': np.float64(-170.48783026537004), 'EBC1:energy_increment': np.float64(-26391372.95479071), 'EBC2:energy_increment': np.float64(-2077157.7265717983)}
Reading Impact_particles = /sdf/data/ad/ard/u/jytang/lume-belt-live/STCAV_data/particle-2025-04-05.h5
Setting BELT BC1:angle = 0.10124061478887932
Setting BELT BC2:angle = 0.04686192354931788
Setting BELT L1:gradient = 15499999.757463621
Setting BELT L1:phase_deg = -19.99214623827802
Setting BELT L2:gradient = 14409695.95218043

<!-- lume-genesis detected Jupyter and will use HTML for rendering. -->

In [24]:
result['outputs']['archive']

'/sdf/data/ad/ard/u/jytang/lume-belt-live/archive/2025/11/03/103a5530aee95e63aa4d8261cabd18c6.h5'

In [ ]:
if __name__ == '__main__':
    while True:
        try:
            result = run1()
            sleep(10)
        except Exception as e:
            logger.info(e)
            if (e.__class__.__name__ == 'Exception'):
                logger.info('Stopping the Program')
                break
            else:
                logger.info('Something BAD happened. Sleeping for 10 s ...')      
                sleep(10)

{'Impact_particles': '/sdf/data/ad/ard/u/jytang/lume-belt-live/STCAV_data/particle-2025-04-05.h5', 'BC1:angle': np.float64(0.10124741090994963), 'BC2:angle': np.float64(0.0468301579344105), 'L1:gradient': np.float64(15500002.020916037), 'L1:phase_deg': np.float64(-25.920255052586402), 'L2:gradient': np.float64(13640737.455305342), 'L2:phase_deg': np.float64(-10.322652212710413), 'L3:gradient': np.float64(13696421.594161749), 'L3:phase_deg': np.float64(0.0), 'HL:gradient': np.float64(10501383.87015447), 'HL:phase_deg': np.float64(-168.6900675259798), 'EBC1:energy_increment': np.float64(-19034034.827237517), 'EBC2:energy_increment': np.float64(-21983678.26046753)}
Reading Impact_particles = /sdf/data/ad/ard/u/jytang/lume-belt-live/STCAV_data/particle-2025-04-05.h5
Setting BELT BC1:angle = 0.10124741090994963
Setting BELT BC2:angle = 0.0468301579344105
Setting BELT L1:gradient = 15500002.020916037
Setting BELT L1:phase_deg = -25.920255052586402
Setting BELT L2:gradient = 13640737.45530534